<a href="https://colab.research.google.com/github/HSA2005-cloud/BIS-Lab/blob/main/3.%20Ant%20Colony%20Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import numpy as np
import random
import math

N = 6
ANTS = 10
MAX_ITER = 100

alpha = 1.0
beta = 2.0
rho = 0.5
Q = 100
initial_pheromone = 0.1

# Packing station and 5 shelves

x = [0, 2, 5, 7, 4, 1]
y = [0, 3, 2, 6, 8, 7]

dist = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        dist[i][j] = math.sqrt(
            (x[i] - x[j])**2 +
            (y[i] - y[j])**2
        )

pheromone = np.full((N, N), initial_pheromone)

def calculate_distance(tour):

    total = 0

    for i in range(len(tour) - 1):
        total += dist[tour[i]][tour[i + 1]]

    return total

def select_next(current, visited):

    probabilities = []
    candidates = []

    for j in range(1, N):

        if not visited[j]:

            value = (
                pheromone[current][j] ** alpha
            ) * (
                (1 / dist[current][j]) ** beta
            )

            probabilities.append(value)
            candidates.append(j)

    total = sum(probabilities)

    probabilities = [
        p / total for p in probabilities
    ]

    return random.choices(
        candidates,
        weights=probabilities,
        k=1
    )[0]

def construct_tour():

    tour = [0]
    visited = [False] * N

    visited[0] = True
    current = 0

    while len(tour) < N:

        next_shelf = select_next(current, visited)

        tour.append(next_shelf)
        visited[next_shelf] = True

        current = next_shelf

    # Return to packing station
    tour.append(0)

    return tour

def update_pheromones(tours, lengths):

    global pheromone

    # Evaporation
    pheromone *= (1 - rho)

    # Pheromone deposition
    for k in range(ANTS):

        deposit = Q / lengths[k]

        tour = tours[k]

        for i in range(len(tour) - 1):

            a = tour[i]
            b = tour[i + 1]

            pheromone[a][b] += deposit
            pheromone[b][a] += deposit

best_distance = float('inf')
best_tour = []

for iteration in range(MAX_ITER):

    tours = []
    lengths = []

    # Each ant constructs a route
    for ant in range(ANTS):

        tour = construct_tour()

        length = calculate_distance(tour)

        tours.append(tour)
        lengths.append(length)

        # Update best route
        if length < best_distance:

            best_distance = length
            best_tour = tour.copy()

    # Update pheromones after all ants finish
    update_pheromones(tours, lengths)

print("\n========== ACO PARAMETERS ==========")

print("Number of Locations:", N)
print("Number of Ants:", ANTS)
print("Maximum Iterations:", MAX_ITER)
print("Alpha (Pheromone Importance):", alpha)
print("Beta (Distance Importance):", beta)
print("Rho (Evaporation Rate):", rho)
print("Q (Pheromone Deposit Constant):", Q)
print("Initial Pheromone:", initial_pheromone)

print("\n========== OPTIMIZATION RESULT ==========")

print("Best Warehouse Picking Route:")

route = []

for shelf in best_tour:

    if shelf == 0:
        route.append("Packing Station")
    else:
        route.append("Shelf " + str(shelf))

print(" -> ".join(route))

print("Minimum Picking Distance:",
      round(best_distance, 2))

print("\nOptimization Completed!")



========== ACO PARAMETERS ==========
Number of Locations: 6
Number of Ants: 10
Maximum Iterations: 100
Alpha (Pheromone Importance): 1.0
Beta (Distance Importance): 2.0
Rho (Evaporation Rate): 0.5
Q (Pheromone Deposit Constant): 100
Initial Pheromone: 0.1

========== OPTIMIZATION RESULT ==========
Best Warehouse Picking Route:
Packing Station -> Shelf 1 -> Shelf 5 -> Shelf 4 -> Shelf 3 -> Shelf 2 -> Packing Station
Minimum Picking Distance: 24.35

Optimization Completed!
